# LLM 08: Structured Outputs

Hands-on:
1. Prompt for JSON and see it fail (the unreliable way)
2. Use JSON mode and see it succeed (the less-bad way)
3. Use Pydantic + schema for validated, typed output (the right way)
4. Retry with validation error feedback
5. Exercises: XML extraction, tool-use style

Deps: `pip install anthropic openai pydantic instructor`

## 1. The Unreliable Way: Ask Nicely for JSON

In [ ]:
import os, json

def call_claude(system, user, **kw):
    from anthropic import Anthropic
    r = Anthropic().messages.create(
        model='claude-sonnet-4-6',
        max_tokens=500, temperature=0,
        system=system, messages=[{'role':'user','content':user}], **kw)
    return r.content[0].text

def call_openai(system, user, **kw):
    from openai import OpenAI
    r = OpenAI().chat.completions.create(
        model='gpt-4o-mini', temperature=0,
        messages=[{'role':'system','content':system},{'role':'user','content':user}], **kw)
    return r.choices[0].message.content

article = '''NovaCore Industries today reported Q3 revenue of $438.2 million,
up 12% year-over-year. Full year guidance was reaffirmed at $1.8B.'''

naive = f'''Extract company, revenue, and guidance from this press release as JSON.

Press release:
{article}'''

if os.getenv('ANTHROPIC_API_KEY'):
    out = call_claude('You extract structured data.', naive)
elif os.getenv('OPENAI_API_KEY'):
    out = call_openai('You extract structured data.', naive)
else:
    out = '[no API key set]'
print(out)
print('\n---')
try:
    parsed = json.loads(out)
    print('parsed:', parsed)
except json.JSONDecodeError as e:
    print('JSON parse error:', e)

## 2. The Less-Bad Way: JSON Mode

Provider-enforced JSON syntax (but not schema). OpenAI / vLLM / Ollama support `response_format="json_object"`.

In [ ]:
if os.getenv('OPENAI_API_KEY'):
    out = call_openai(
        'You extract structured data and respond ONLY with valid JSON.',
        naive,
        response_format={'type': 'json_object'},
    )
    print(out)
    print('parsed:', json.loads(out))
else:
    print('Skipped (OpenAI JSON-mode requires OPENAI_API_KEY).')

## 3. The Right Way: Pydantic + JSON Schema

Schema enforcement means *the model's decoder cannot emit an invalid token*.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PressReleaseExtract(BaseModel):
    company: str = Field(description='Legal name of the company')
    revenue_usd: float = Field(description='Quarterly revenue in USD', ge=0)
    guidance_usd: Optional[float] = Field(default=None, description='Full-year guidance in USD, if stated')
    yoy_growth_pct: Optional[float] = Field(default=None, description='Year-over-year growth percent, if stated')

schema = PressReleaseExtract.model_json_schema()
print('JSON Schema:')
print(json.dumps(schema, indent=2)[:500], '...')

In [ ]:
# OpenAI structured outputs (strict)
if os.getenv('OPENAI_API_KEY'):
    from openai import OpenAI
    client = OpenAI()
    r = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role':'system','content':'Extract the structured fields from the press release.'},
            {'role':'user','content':article},
        ],
        response_format=PressReleaseExtract,
    )
    parsed: PressReleaseExtract = r.choices[0].message.parsed
    print(parsed)
    print('\ntyped revenue:', parsed.revenue_usd, type(parsed.revenue_usd))
else:
    print('Skipped (needs OPENAI_API_KEY).')

## 4. Claude Tool Use (Typed Arguments)

Claude's `tool_use` returns validated arguments as a typed dict.

In [ ]:
if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    client = Anthropic()
    tool = {
        'name': 'extract_press_release',
        'description': 'Extract structured fields from a press release',
        'input_schema': schema,
    }
    r = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=500,
        tools=[tool],
        tool_choice={'type': 'tool', 'name': 'extract_press_release'},
        messages=[{'role':'user','content':article}],
    )
    block = next(b for b in r.content if b.type == 'tool_use')
    print('tool input (typed dict):', block.input)
    parsed = PressReleaseExtract(**block.input)
    print('pydantic parsed:', parsed)
else:
    print('Skipped (needs ANTHROPIC_API_KEY).')

## 5. Retry with Validation Feedback (Instructor Pattern)

In [ ]:
from pydantic import ValidationError

def extract_with_retry(article: str, model_cls, max_retries: int = 2):
    last_error = None
    for attempt in range(max_retries + 1):
        prompt = article if attempt == 0 else (
            f"{article}\n\nYour previous output failed validation with: {last_error}\n"
            "Return valid JSON matching the schema."
        )
        raw = call_openai('Return only JSON matching the schema.', prompt,
                          response_format={'type':'json_object'})
        try:
            return model_cls.model_validate_json(raw)
        except (ValidationError, json.JSONDecodeError) as e:
            last_error = str(e)
    raise RuntimeError(f'Failed after {max_retries+1} attempts: {last_error}')

if os.getenv('OPENAI_API_KEY'):
    print(extract_with_retry(article, PressReleaseExtract))

## 6. Exercise: XML Tag Extraction (Claude-style)

Write a system prompt asking Claude to return:

```
<company>...</company>
<revenue>...</revenue>
<guidance>...</guidance>
```

Then parse with `re.search(r'<company>(.*?)</company>', out, re.S)`. Measure success rate on 20 representative press releases. You'll usually find XML-tag reliability is nearly as good as schema mode for simple extraction, with less setup.

## 7. Exercise: Batch Extraction with Failure Accounting

Process a list of 50 press releases. Track: (a) schema violations, (b) semantic errors detected by rule-based checks, (c) latency distribution. Plot the success rate by retry count. This is the shape of every production extraction pipeline.

## 8. Takeaways

- **Provider-enforced schemas are the gold standard.** Use them.
- **Pydantic is the data contract** between LLM and your app.
- **Retry with validation error feedback** handles the last 1–5%.
- **Never parse by regex** what a schema can validate.
- **Track structured-output success rate as an SLI**, not just latency.

Next: **09 — Context Windows & Long Context**, where we study the one resource that constrains everything.